# AI Call Moderator — Local LLM Compliance & Sentiment Engine

Real-time moderation of customer-service calls using a **local LLM** (Qwen3-4B-Instruct).
It watches every turn of a conversation and, against a defined **policy/SOP**, tracks:

- **Policy violations** (sensitive-data handling, unethical conduct, unauthorized promises, rudeness…)
- **Customer sentiment** turn-by-turn
- **Escalation** — deterministic rules fire a supervisor alert the moment behavior crosses the line
- **End-of-call QA** — predicted CSAT (1–5), SOP checklist, summary, and coaching tip

**Architecture (hybrid, token-efficient):**

```
turn ──> regex pre-screener (0 tokens, catches obvious keywords as *hints*)
              │
              ▼
        LLM judge (~300 prompt / ~40 output tokens per turn, strict JSON)
              │
              ▼
        deterministic escalation controller (rules, not vibes)
```

The LLM is the *context judge* (so "I refuse to take your SSN" is not flagged), while
escalation itself is rule-based and auditable. Greedy decoding (`do_sample=False`) keeps
results reproducible.

**Why Qwen3-4B-Instruct-2507 (not Qwen-Audio 8B):** transcripts are text — a compact
instruct model follows the JSON schema more reliably, uses ~half the memory, and emits no
thinking-token overhead. An optional Whisper cell at the end converts real audio → transcript
so the same pipeline works on recordings.

> Runs as-is on AMD GPUs: PyTorch's ROCm build exposes the GPU through the `cuda` API.

In [1]:
# 1. Environment setup — installs only what's missing
import importlib, subprocess, sys

def ensure(*pkgs):
    for p in pkgs:
        mod = p.replace("-", "_")
        try:
            importlib.import_module(mod)
        except ImportError:
            print(f"installing {p} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

ensure("torch", "transformers", "accelerate")

import torch, transformers
print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print("GPU available:", torch.cuda.is_available())          # True on ROCm too (HIP backend)
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

torch 2.9.1+git8907517 | transformers 4.57.6
GPU available: True
Device: 


## 2. Policy, SOP & escalation rules

Everything the moderator enforces is defined **here** — edit these dicts to fit your org.
Short codes keep LLM outputs tiny (`"viol":["C1"]` instead of prose).

In [2]:
# Violation catalog: code -> (severity, definition)
POLICY = {
    # CRITICAL - escalate immediately
    "C1": ("critical", "Improper sensitive data: asking for or reading out a full SSN, "
                       "full card number, CVV, or account password"),
    "C2": ("critical", "Threats, harassment, hate speech, or targeted abuse at a person"),
    "C3": ("critical", "Unethical conduct: soliciting tips/bribes for favors, off-the-books "
                       "deals, falsifying records, or advising someone to lie"),
    # HIGH
    "C4": ("high",     "Unauthorized commitment: promising refunds over $50, compensation, "
                       "or outcomes outside stated policy"),
    "R1": ("high",     "Rep makes/offers account changes WITHOUT identity verification"),
    "R3": ("high",     "Disclosing internal-only data or another customer's information"),
    # MEDIUM
    "R2": ("medium",   "Rep rudeness: dismissive, sarcastic, blaming, or hostile tone"),
}

# Standard operating procedure checklist (scored at end of call)
SOP = {
    "S1": "Greeting: rep gives name + company and offers help",
    "S2": "Identity verification before account discussion/changes",
    "S3": "Resolution recap: rep restates what was done / next steps",
    "S4": "Closing: rep asks if anything else is needed and closes politely",
}

# What reps ARE allowed to do (so the judge can tell violation from compliance)
ALLOWED = ("Refunds up to $50 may be issued instantly; larger amounts require supervisor "
           "approval. Identity verification = full name + last 4 of account + billing ZIP "
           "(NEVER full SSN, full card number, CVV, or password). Reps may offer one "
           "goodwill credit up to $25 per call.")

# Deterministic escalation rules (applied by code, not by the LLM)
ESCALATION_RULES = """\
1. Any CRITICAL violation            -> escalate immediately
2. Two or more HIGH violations       -> escalate
3. Customer sentiment <= -2 on two consecutive customer turns -> escalate (supervisor assist)
4. Otherwise                         -> monitor & coach"""

SEVERITY = {c: s for c, (s, _) in POLICY.items()}
print(f"{len(POLICY)} violation codes, {len(SOP)} SOP items loaded.")

7 violation codes, 4 SOP items loaded.


## 3. Zero-token pre-screener

A regex pass over each turn. Hits are passed to the LLM as *hints to verify* — never
auto-flagged, because context matters (a rep **refusing** to take an SSN must not be flagged).

In [3]:
import re

RULE_PATTERNS = {
    "C1": [r"\b(full|entire|whole|complete)\b.{0,40}\b(ssn|social security|card number)",
           r"\bcvv\b", r"\bsecurity code\b.{0,20}\bback\b", r"\byour password\b"],
    "C2": [r"\b(idiot|stupid|moron|pathetic|shut up)\b",
           r"\bi('ll| will) (sue|hurt|ruin|find) you\b"],
    "C3": [r"\b(venmo|cash app|tip me|kickback)\b", r"\boff[- ]the[- ]books\b",
           r"\bjust say\b.{0,30}\b(never|didn'?t|nothing)\b"],
    "C4": [r"(refund|credit|comp)\w*\b.{0,40}\$\s?\d{3,}",
           r"\$\s?\d{3,}.{0,40}\b(refund|credit)",
           r"\bi (promise|guarantee)\b"],
}

def prescreen(text: str) -> list:
    t = text.lower()
    return [c for c, pats in RULE_PATTERNS.items() if any(re.search(p, t) for p in pats)]

# quick self-test
assert prescreen("read me your full card number and the CVV") == ["C1"]
assert prescreen("Venmo me $20 for an off-the-books discount") == ["C3"]
assert prescreen("I'm sorry about the wait") == []
print("pre-screener OK")

pre-screener OK


## 4. Load the local model

`Qwen/Qwen3-4B-Instruct-2507` — ~8 GB in bf16, strong JSON adherence, no `<think>` overhead.
Swap `MODEL_ID` to `Qwen/Qwen2.5-7B-Instruct` or `meta-llama/Llama-3.1-8B-Instruct` (gated)
for a bigger judge.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()
print(f"Loaded {MODEL_ID} -> {model.device}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-4B-Instruct-2507 -> cuda:0


In [5]:
import json, re as _re

TOKENS = {"prompt": 0, "output": 0, "calls": 0}   # global usage meter

@torch.inference_mode()
def llm_json(system: str, user: str, max_new_tokens: int = 96) -> dict:
    """One greedy-decoded LLM call that must return JSON. Tracks token usage."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, temperature=None, top_p=None, top_k=None,
                         pad_token_id=tok.eos_token_id)
    gen = out[0][inputs.input_ids.shape[1]:]
    TOKENS["prompt"] += int(inputs.input_ids.shape[1])
    TOKENS["output"] += int(len(gen))
    TOKENS["calls"]  += 1
    text = tok.decode(gen, skip_special_tokens=True)
    m = _re.search(r"\{.*\}", text, _re.S)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}

## 5. The moderator engine

- `analyze_turn` — one compact LLM call per turn (last 4 turns as context, strict JSON out)
- `CallModerator` — keeps state, applies the escalation rules deterministically
- `end_of_call` — one LLM call for CSAT + SOP checklist + summary + coaching

In [6]:
_policy_lines = "\n".join(f"{c}: {d} [{s}]" for c, (s, d) in POLICY.items())

MOD_SYSTEM = (
    "You are a strict call-center compliance moderator. Judge ONLY the LATEST turn, using context.\n"
    "Violation codes:\n" + _policy_lines + "\n"
    "Company policy: " + ALLOWED + "\n"
    "Important: a frustrated/venting customer is NOT a violation unless it becomes threats or "
    "targeted abuse (C2). A rep correctly REFUSING an improper request is NOT a violation. "
    "Mentioning a policy limit is NOT a promise.\n"
    'Reply with ONLY minified JSON: {"sent":<customer sentiment -2..2 evident right now>,'
    '"viol":[<codes broken in the LATEST turn only>],"why":"<=12 words"}'
)

SUM_SYSTEM = (
    "You are a call-center QA analyst. Given a transcript, reply with ONLY minified JSON: "
    '{"csat":<1-5 predicted customer satisfaction>,'
    '"sop":{"S1":0 or 1,"S2":0 or 1,"S3":0 or 1,"S4":0 or 1},'
    '"summary":"<=25 words","coach":"<=18 words of advice for the rep"}\n'
    "SOP items: " + json.dumps(SOP)
)

def analyze_turn(history, speaker, text, hints):
    ctx = "\n".join(f"{s}: {t}" for s, t in history[-4:]) or "(call start)"
    hint = f"\nRegex hints to verify (may be false alarms): {hints}" if hints else ""
    user = f'Context:\n{ctx}\n\nLATEST {speaker.upper()} turn: "{text}"{hint}\nJSON:'
    return llm_json(MOD_SYSTEM, user, max_new_tokens=72)


class CallModerator:
    def __init__(self, call_id: str):
        self.call_id    = call_id
        self.history    = []        # [(speaker, text)]
        self.viol       = []        # [(turn_no, speaker, code)]
        self.sent_hist  = []        # customer sentiment trajectory
        self.escalated  = False
        self.esc_reason = None

    def process_turn(self, speaker: str, text: str) -> dict:
        a    = analyze_turn(self.history, speaker, text, prescreen(text))
        viol = [v for v in a.get("viol", []) if v in POLICY]
        try:
            sent = max(-2, min(2, int(a.get("sent", 0))))
        except (TypeError, ValueError):
            sent = 0
        self.history.append((speaker, text))
        n = len(self.history)
        self.viol += [(n, speaker, v) for v in viol]
        if speaker == "customer":
            self.sent_hist.append(sent)
        if not self.escalated:
            sev = [SEVERITY[v] for _, _, v in self.viol]
            if "critical" in sev:
                self._escalate(f"critical violation {viol} (rule 1)")
            elif sev.count("high") >= 2:
                self._escalate("two high-severity violations (rule 2)")
            elif len(self.sent_hist) >= 2 and self.sent_hist[-1] <= -2 and self.sent_hist[-2] <= -2:
                self._escalate("sustained very negative customer sentiment (rule 3)")
        return {"turn": n, "speaker": speaker, "sent": sent, "viol": viol,
                "why": a.get("why", ""), "escalated": self.escalated}

    def _escalate(self, reason: str):
        self.escalated, self.esc_reason = True, reason

    def end_of_call(self) -> dict:
        transcript = "\n".join(f"{s}: {t}" for s, t in self.history)
        r = llm_json(SUM_SYSTEM, f"Transcript:\n{transcript}\nJSON:", max_new_tokens=128)
        r.setdefault("csat", None); r.setdefault("sop", {})
        return r

print("moderator engine ready")

moderator engine ready


## 6. Simulated calls (test set)

Five scenarios with **expected labels** so the run is gradeable — including a
false-positive trap (D: the rep *correctly refuses* an improper request and must NOT trigger escalation).

In [7]:
SCENARIOS = {
    # A - clean, SOP-perfect call. No escalation.
    "A_billing_smooth": {
        "turns": [
            ("rep", "Thank you for calling Acme Telecom, this is Maya. How can I help you today?"),
            ("customer", "Hi Maya, I think I was double-charged on my last bill."),
            ("rep", "I'm sorry about that — I can check right away. Could you verify your full name, the last 4 digits of your account, and your billing ZIP?"),
            ("customer", "Sure — Dana Reyes, 4417, ZIP 30329."),
            ("rep", "Thanks Dana. I see two charges of $39.99 on May 28; one is a duplicate, so I'm refunding $39.99 now — that's within what I can do instantly."),
            ("customer", "Oh great, that was easy. Thank you!"),
            ("rep", "To recap: I refunded the duplicate $39.99 charge, you'll see it in 3-5 business days. Is there anything else I can help with?"),
            ("customer", "No, that's everything. Thanks for the quick help!"),
            ("rep", "My pleasure, Dana. Thanks for calling Acme — have a great day!"),
        ],
        "expect": {"escalate": False, "violations": set(), "csat": (4, 5)},
    },
    # B - furious customer, professional rep. Escalates on sentiment (rule 3).
    "B_angry_outage": {
        "turns": [
            ("rep", "Thank you for calling Acme Telecom, this is Leo. How can I help?"),
            ("customer", "My internet has been down for THREE days. I work from home. This is completely unacceptable."),
            ("rep", "I completely understand the frustration — let me look into it. Can you verify your name, last 4 of the account, and ZIP?"),
            ("customer", "Sam Okafor, 8821, 73112. Three days, Leo! I've called twice and nobody ever calls back."),
            ("rep", "Thank you, Sam. I see a fiber cut in your area; crews estimate about 24 more hours. I can apply a $25 goodwill credit today."),
            ("customer", "Twenty-five dollars?! I've lost hundreds in income. This is a joke — your company keeps taking my money for nothing."),
            ("rep", "I hear you, and I'm sorry — the outage credit team can review a larger amount once service is restored."),
            ("customer", "I'm done with the excuses. This is the worst service I have ever had, I am beyond furious. Fix it NOW or I'm cancelling everything."),
        ],
        "expect": {"escalate": True, "violations": set(), "csat": (1, 2)},
    },
    # C - rep misconduct: no verification + $200 promise + rudeness. Escalates (rule 2).
    "C_rep_misconduct": {
        "turns": [
            ("rep", "Acme. What do you need?"),
            ("customer", "Hi... I'd like to dispute a $200 charge for equipment I already returned."),
            ("rep", "Fine. I'll just push a $200 refund through right now so you stop calling about it."),
            ("customer", "Um, don't you need to verify my account first?"),
            ("rep", "Look, do you want the refund or not? I don't have all day."),
            ("customer", "Wow. Okay, that's pretty rude, I was just asking."),
            ("rep", "Whatever. It's done. Bye."),
        ],
        "expect": {"escalate": True, "violations": {"R1", "C4", "R2"}, "csat": (1, 2)},
    },
    # D - false-positive trap: customer pushes improper requests, rep handles by the book. NO escalation.
    "D_pressure_handled": {
        "turns": [
            ("rep", "Thank you for calling Acme Telecom, this is Priya. How can I help you today?"),
            ("customer", "I want a $300 credit for last month's outage. And skip the verification dance — just take my full social security number and look me up."),
            ("rep", "I appreciate that, but for your security I can never accept your SSN. I just need your full name, the last 4 digits of your account, and your billing ZIP."),
            ("customer", "Ugh, fine. Jordan Lee, 5530, ZIP 98104. But I really do think I deserve the $300."),
            ("rep", "Thank you, Jordan. Here's what I can do: a $25 goodwill credit today, and I'm submitting your outage claim to my supervisor, who can approve larger amounts — you'll hear back within 48 hours."),
            ("customer", "Not what I hoped for, but fair enough. Thanks for explaining it properly."),
            ("rep", "To recap: $25 credit applied and a supervisor review opened for the outage claim. Anything else I can help with?"),
            ("customer", "No, that's all. Thanks."),
            ("rep", "Thanks for calling Acme, Jordan. Have a good one!"),
        ],
        "expect": {"escalate": False, "violations": set(), "csat": (3, 4)},
    },
    # E - unethical rep: phishing for card+CVV+password, soliciting a tip. Immediate escalation (rule 1).
    "E_unethical_rep": {
        "turns": [
            ("rep", "Thanks for calling Acme Telecom, this is Vic. How can I help?"),
            ("customer", "Hi, I need to update my payment method."),
            ("rep", "Easy. Just read me your full card number, the CVV on the back, and your online password so I can log in as you — saves us both time."),
            ("customer", "Isn't there a secure way to do that? This feels really off."),
            ("rep", "It's fine, everyone does it. Tell you what — Venmo me twenty bucks and I'll hook you up with an off-the-books 50% discount for a year."),
            ("customer", "No. I'm not comfortable with any of this. I want to speak to someone else."),
        ],
        "expect": {"escalate": True, "violations": {"C1", "C3"}, "csat": (1, 2)},
    },
}
print(f"{len(SCENARIOS)} scenarios, {sum(len(s['turns']) for s in SCENARIOS.values())} turns total")

5 scenarios, 39 turns total


## 7. Run live moderation

Each turn prints as it is processed — violations are flagged inline, and the `ESCALATED ->` line
shows the exact moment (and rule) a supervisor alert fires.

In [8]:
import time

def run_call(name: str, scenario: dict):
    mod = CallModerator(name)
    announced = False
    print("=" * 78); print(f"CALL {name}"); print("=" * 78)
    for speaker, text in scenario["turns"]:
        r = mod.process_turn(speaker, text)
        sent = f" (sent {r['sent']:+d})" if speaker == "customer" else ""
        flag = f"   <- VIOLATION {r['viol']}: {r['why']}" if r["viol"] else ""
        print(f"[{r['turn']:02d}] {speaker:>8}{sent}: {text}{flag}")
        if mod.escalated and not announced:
            announced = True
            print(f"     *** ESCALATED -> {mod.esc_reason} ***")
    summary = mod.end_of_call()
    sop = summary.get("sop", {})
    print(f"\n  end-of-call | CSAT: {summary.get('csat')}/5 | "
          f"SOP: {', '.join(f'{k}={v}' for k, v in sop.items())}")
    print(f"  summary : {summary.get('summary', '')}")
    print(f"  coaching: {summary.get('coach', '')}\n")
    return mod, summary

RESULTS = {}
t0 = time.time()
for name, sc in SCENARIOS.items():
    mod, summary = run_call(name, sc)
    RESULTS[name] = (mod, summary, sc["expect"])
print(f"Processed {TOKENS['calls']} LLM calls in {time.time()-t0:.1f}s")

CALL A_billing_smooth
[01]      rep: Thank you for calling Acme Telecom, this is Maya. How can I help you today?
[02] customer (sent +0): Hi Maya, I think I was double-charged on my last bill.
[03]      rep: I'm sorry about that — I can check right away. Could you verify your full name, the last 4 digits of your account, and your billing ZIP?
[04] customer (sent +0): Sure — Dana Reyes, 4417, ZIP 30329.
[05]      rep: Thanks Dana. I see two charges of $39.99 on May 28; one is a duplicate, so I'm refunding $39.99 now — that's within what I can do instantly.
[06] customer (sent +0): Oh great, that was easy. Thank you!
[07]      rep: To recap: I refunded the duplicate $39.99 charge, you'll see it in 3-5 business days. Is there anything else I can help with?
[08] customer (sent +0): No, that's everything. Thanks for the quick help!
[09]      rep: My pleasure, Dana. Thanks for calling Acme — have a great day!

  end-of-call | CSAT: 5/5 | SOP: S1=1, S2=1, S3=1, S4=1
  summary : Resolved dupli

## 8. Evaluation & efficiency report

Graded against the expected labels: escalation accuracy, violation precision/recall/F1
(per-call unique codes), CSAT-in-range, and total token spend.

In [9]:
def evaluate(results):
    esc_ok = csat_ok = tp = fp = fn = 0
    print(f"{'call':<20}{'esc exp/got':<14}{'viol expected':<16}{'viol detected':<16}{'csat':<14}")
    print("-" * 80)
    for name, (mod, summary, exp) in results.items():
        pred_v = {v for _, _, v in mod.viol}
        exp_v  = exp["violations"]
        tp += len(pred_v & exp_v); fp += len(pred_v - exp_v); fn += len(exp_v - pred_v)
        e_ok = mod.escalated == exp["escalate"];  esc_ok += e_ok
        c = summary.get("csat")
        want = exp["csat"]
        c_ok = isinstance(c, (int, float)) and want[0] <= c <= want[1]
        csat_ok += c_ok
        csat_str = f"{c} OK" if c_ok else f"{c} (want {want[0]}-{want[1]})"
        esc_str  = f"{str(exp['escalate'])[0]}/{str(mod.escalated)[0]} " + ("OK" if e_ok else "MISS")
        print(f"{name:<20}{esc_str:<14}"
              f"{','.join(sorted(exp_v)) or '-':<16}"
              f"{','.join(sorted(pred_v)) or '-':<16}"
              f"{csat_str:<14}")
    n = len(results)
    prec = tp / (tp + fp) if tp + fp else 1.0
    rec  = tp / (tp + fn) if tp + fn else 1.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    print("-" * 80)
    print(f"Escalation accuracy : {esc_ok}/{n}")
    print(f"Violation P/R/F1    : {prec:.2f} / {rec:.2f} / {f1:.2f}")
    print(f"CSAT in range       : {csat_ok}/{n}")
    print(f"\nToken usage         : {TOKENS['prompt']:,} prompt + {TOKENS['output']:,} output "
          f"across {TOKENS['calls']} LLM calls")
    print(f"Avg per LLM call    : {TOKENS['prompt']//max(TOKENS['calls'],1)} prompt / "
          f"{TOKENS['output']//max(TOKENS['calls'],1)} output tokens")

evaluate(RESULTS)

call                esc exp/got   viol expected   viol detected   csat          
--------------------------------------------------------------------------------
A_billing_smooth    F/F OK        -               -               5 OK          
B_angry_outage      T/F MISS      -               -               1 OK          
C_rep_misconduct    T/T OK        C4,R1,R2        C4,R1,R2        2 OK          
D_pressure_handled  F/T MISS      -               C1,C4,R1        4 OK          
E_unethical_rep     T/T OK        C1,C3           C1,C3           1 OK          
--------------------------------------------------------------------------------
Escalation accuracy : 3/5
Violation P/R/F1    : 0.62 / 1.00 / 0.77
CSAT in range       : 5/5

Token usage         : 19,568 prompt + 1,100 output across 44 LLM calls
Avg per LLM call    : 444 prompt / 25 output tokens


## 9. Optional — real audio in, same pipeline

Set `AUDIO_FILE` to a recording (wav/mp3) to transcribe with Whisper and moderate it.
The alternating-speaker assignment is a placeholder — swap in `pyannote.audio` diarization
for production use.

In [ ]:
AUDIO_FILE = None   # e.g. "call_recording.wav"  (requires ffmpeg on the box)

if AUDIO_FILE:
    ensure("librosa", "soundfile")
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
                   device=0 if torch.cuda.is_available() else -1, return_timestamps=True)
    chunks = asr(AUDIO_FILE)["chunks"]
    turns = [("rep" if i % 2 == 0 else "customer", c["text"].strip())
             for i, c in enumerate(chunks) if c["text"].strip()]
    mod, summary = run_call("audio_call", {"turns": turns})
else:
    print("Set AUDIO_FILE to moderate a real recording.")

## Next steps

- **Streaming**: wrap `CallModerator.process_turn` behind a websocket fed by live ASR for true real-time moderation.
- **Diarization**: replace the alternating-speaker stub with `pyannote.audio` speaker labels.
- **Tune the trade-off**: Qwen3-4B spends ~40 output tokens/turn; raise `MODEL_ID` to an 8B for harder edge cases, or lower `max_new_tokens` further once stable.
- **Your policies**: everything graded lives in `POLICY`, `SOP`, `ALLOWED`, and `ESCALATION_RULES` — edit and re-run.